In [ ]:
# Standard packages

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

# Model

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split, StratifiedGroupKFold
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.calibration import CalibratedClassifierCV

# Pipeline + Transformer:

from sklearn.preprocessing import OneHotEncoder, StandardScaler,OrdinalEncoder, TargetEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

# Neural network

import torch
from torch import nn
#import cudf

# Speed up options
#%load_ext cudf.pandas
#%load_ext cuml.accel



In [ ]:
# Reading in data

train = pd.read_csv('train.csv')

test = pd.read_csv('test.csv')

orig = pd.read_csv('f1_strategy_dataset_v4.csv')

changes = pd.read_csv('changes_made.csv')

scores_df = pd.read_csv('scores_df.csv')


In [ ]:
# Setting up target columns and test IDs

ytrain = train['PitNextLap']

yorig = orig['PitNextLap']

test_id = test['id']

In [ ]:
# Removing ID column and target

Xtrain = train.drop(columns=['id','PitNextLap'], axis = 1).copy()

Xtest = test.drop(columns = ['id'], axis = 1).copy()

Xorig = orig.drop(columns= ['Normalized_TyreLife', 'PitNextLap'], axis = 1).copy()

# Data cleaning #

In [ ]:


# Null values

Xtrain.info(), Xtest.info() , Xorig.info() # No null values


In [ ]:
num_cols = Xtrain.select_dtypes(['int','float']).columns.to_list()
num_cols

In [ ]:
# Understanding if any feature outliers have a significant impact on target

sns.displot(
    data=train,
    x = 'TyreLife',
    col = 'PitNextLap',
    height = 5,
    bins = 'fd'
)

sns.displot(
    data=train,
    x = 'Stint',
    col = 'PitNextLap',
    height = 5,
    bins = 'fd',
    discrete = True
)

sns.displot(
    data=train,
    x = 'LapNumber',
    col = 'PitNextLap',
    height = 5,
    bins = 'fd',
    discrete = True
)


sns.displot(
    data=train,
    x = 'Position',
    col = 'PitNextLap',
    height = 5,
    bins = 'fd',
    discrete = True
)

sns.displot(
    data=train,
    x = 'LapTime (s)',
    col = 'PitNextLap',
    height = 5,
    bins = 'fd'
)

sns.displot(
    data=train,
    x = 'LapTime_Delta',
    col = 'PitNextLap',
    height = 5,
    bins = 'fd'
)


sns.displot(
    data=train,
    x = 'Cumulative_Degradation',
    col = 'PitNextLap',
    height = 5,
    bins = 'fd',
    discrete = True
)

sns.displot(
    data=train,
    x = 'RaceProgress',
    col = 'PitNextLap',
    height = 5,
    bins = 'fd',
    discrete = True
)

sns.displot(
    data=train,
    x = 'Position_Change',
    col = 'PitNextLap',
    height = 5,
    bins = 'fd',
    discrete = True
)


In [ ]:
sns.displot(
    x = 'Year',
    data = train,
    col = 'PitNextLap'
)

In [ ]:
# Summary statisitcs
Xtrain[num_cols].describe()


In [ ]:
# Winsorization
cols_to_remove = ['PitStop','PitNextLap'] # Binary columns that contain imbalanced responses

def num_outliers(df, choice):

    """
    The function applies the moethod of winsorization to outliers. Choice is a boolean repsonse. True applies the method to the data
    frame. Fasle just prints out any information you would like to know about the data frame 
    """

    df_new = df.copy()

    num_cols = df_new.select_dtypes(['int','float']).columns.to_list()

    num_cols = list(set(num_cols) - set(cols_to_remove))

    df_new[num_cols] = df_new[num_cols].astype('float')

    for col in num_cols:

        q1 = np.quantile(df_new[col], 0.25)

        q3 = np.quantile(df_new[col], 0.75)

        iqr = q3 - q1

        lower = q1 - 1.5 * iqr

        upper = q3 + 1.5 * iqr

        # Mask returns true/falses where data column meets outlier criteria

        mask = (df_new[col] < lower) | (df_new[col] > upper)

        low_mask = (df_new[col] < lower)

        high_mask = (df_new[col] > upper)

        n_outliers = mask.sum()

        print(f'{col} has {n_outliers} outliers')

        if choice == True: # Winsorization 

            df_no_outliers = df_new.loc[~mask]

            df_out_low = df_new.loc[low_mask]

            df_out_high = df_new.loc[high_mask]

            df_out_high.loc[:, col] = upper

            df_out_low.loc[:, col] = lower

            df_new = pd.concat([df_no_outliers, df_out_high, df_out_low], axis = 0).sort_index()
        else:
            print('No changes made to dataframe')



    print(f'Shape of new data frame {df_new.shape} | Shape of old data frame {df.shape}')


    return df_new



In [ ]:
Xtrain_outliers = num_outliers(Xtrain, True)
print('//////////')
Xtest_outliers = num_outliers(Xtest, True)
print('//////////')
Xorig_outliers = num_outliers(Xorig, True)

In [ ]:
# Seeing how the distribution of the numerical columns changes after function is applied
Xtrain_outliers[num_cols].describe()

In [ ]:
cat_cols = Xtrain_outliers.select_dtypes('object').columns.to_list()
cat_cols

In [ ]:
# Identifying any potential columns with categorical outliers or anomalies, mis-entries etc. and what columns to encode
Xtrain_outliers[cat_cols].describe()

In [ ]:
Xtrain_outliers['Driver'].unique()

In [ ]:
Xtrain_outliers['Driver'].str.startswith('D1')

In [ ]:
# Categorical outliers

def cat_outliers(df):

    df_new = df.copy()

    for i in range(10):

        df_new['Driver'] = df_new['Driver'].apply(lambda x: f'D{i}00s' if x.startswith(f'D{i}') else x)


    print('All categorical outliers were removed')

    return df_new




In [ ]:
Xtrain_cat_outliers = cat_outliers(Xtrain_outliers)
Xtest_cat_outliers = cat_outliers(Xtest_outliers)
Xorig_cat_outliers = cat_outliers(Xorig_outliers)

# Feature engineering #

In [ ]:
# Categorical feature engineering

def cat_fe(df):

    df_new = df.copy()

    for i, c1 in enumerate(cat_cols):
        for j, c2 in enumerate(cat_cols):
            if i < j:

                df_new[f'BI_Combin_{c1}_{c2}'] = df_new[c1].astype(str) + "_" + df_new[c2].astype(str)

    for i, c1 in enumerate(cat_cols):
        for j, c2 in enumerate(cat_cols):
            for k, c3 in enumerate(cat_cols):
                if i < j:
                    if j < k:

                        df_new[f'TRI_Combin_{c1}_{c2}_{c3}'] = df_new[c1].astype(str) + "_" + df_new[c2].astype(str) + "_" + df_new[c3].astype(str)



                

    print('Categorical Feature Engineering complete')

    return df_new


In [ ]:
Xtrain_cat = cat_fe(Xtrain_cat_outliers)

Xtest_cat = cat_fe(Xtest_cat_outliers)

Xorig_cat = cat_fe(Xorig_cat_outliers)

In [ ]:
# Numerical feature engineering

def num_fe(df):

    df_new = df.copy()

    df_new['Lap_x_Delta'] = df_new['LapTime (s)'] * df_new['LapTime_Delta']

    df_new['Average_Position_Change'] = (df_new['Position'] + df_new['Position_Change'])/2

    df_new['TyreLife_Left'] = df_new['TyreLife']/(df_new['LapNumber'])

    df_new['Stint_Per_Year'] =   df_new['Stint']/df_new['Year']

    df_new['Cumulative_Degradation_Outliers'] = np.absolute(df_new['Cumulative_Degradation'] - df_new['Cumulative_Degradation'].mean())

    df_new['LapTimeDelta_Per_Year'] = df_new['LapTime_Delta']/df_new['Year']

    df_new['CD_Per_Year'] = df_new['Cumulative_Degradation']/df_new['Year']

    df_new['Tyre_To_Progress'] = df_new['TyreLife'] * df_new['RaceProgress']

    df_new['Tyre_To_Lap'] = df_new['TyreLife'] * df_new['LapNumber']

    df_new['Stint_Per_Delta'] = df_new['Stint']/(df_new['LapTime_Delta'] + df_new['LapTime_Delta'].min())


    print('Numerical Feature Engineering is complete')


    return df_new



In [ ]:
Xtrain_num = num_fe(Xtrain_cat)

Xtest_num = num_fe(Xtest_cat)

Xorig_num = num_fe(Xorig_cat)

In [ ]:
# Removing rows containing races not included in test or training set

mask = (Xorig_num['Race'] == 'Pre-Season Test') | (Xorig_num['Race'] == 'Pre-Season Track Session' )

Xorig_num = Xorig_num[~mask].reset_index(drop=True)

yorig = yorig.iloc[Xorig_num.index]


In [ ]:
# Frequency encoding
def freq_encoder(df):

    df_new = df.copy()

    cat_cols = df_new.select_dtypes('object').columns.to_list()

    drop_cols = list(set(cat_cols) - set(['Compound','Driver','Race']))

    for col in cat_cols:

        freq_tab = pd.concat([Xtrain_num[col], Xtest_num[col], Xorig_num[col]]).value_counts()

        df_new[f'Freq_{col}'] = df_new[col].map(freq_tab).fillna(0).astype(float)

    print('Frequency encoding is complete')


    df_new = df_new.drop(columns = drop_cols, axis = 1)

    return df_new



In [ ]:
Xtrain_freq = freq_encoder(Xtrain_num)

Xtest_freq = freq_encoder(Xtest_num)

Xorig_freq = freq_encoder(Xorig_num)

In [ ]:
Xtrain_freq.info()

In [ ]:


num_pro = Xtrain_freq.select_dtypes(['int','float']).columns.to_list()

ohe_pro = ['Compound']

te_pro = ['Driver', 'Compound','Race','Year']
te_pro


# Preprocessing #

In [ ]:
preprocessor = ColumnTransformer([
    ('Num', Pipeline([
        ('scaler', StandardScaler())
    ]), num_pro),

    ('Cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(drop='first', sparse_output=False))
    ]), ohe_pro)
])

def data_prep(df):

    df_new = df.copy()

    ImputedX = preprocessor.fit_transform(Xtrain_freq)

    colnames = preprocessor.get_feature_names_out()

    df_imputed = preprocessor.transform(df_new)

    df_final = pd.DataFrame(df_imputed, columns=colnames, index=df.index)

    df_final = df_final.merge(df.loc[:, ['Race', 'Year','Compound','Driver']].astype('category'), left_index=True, right_index=True)


    print('Data preprocessing is complete')

    return df_final

In [ ]:
Xtrain_pre = data_prep(Xtrain_freq)

Xtest_pre = data_prep(Xtest_freq)

Xorig_pre = data_prep(Xorig_freq)

In [ ]:
Xtest_pre.shape, Xorig_pre.shape, Xtrain_pre.shape, ytrain.shape, yorig.shape

In [ ]:
Xtrain_freq[num_cols].aggregate(np.var, axis=0).sort_values()

In [ ]:
Xtrain_final = pd.concat([Xtrain_pre,Xorig_pre], axis = 0, ignore_index=True)

ytrain_final = pd.concat([ytrain,yorig], axis = 0, ignore_index=True)


Xtrain_final.shape, ytrain_final.shape, Xtest_pre.shape

In [ ]:
# PCA feature engineering 

def pca_fe(df):

    df_new = df.copy()

    pca_cols = df_new.columns[df_new.columns.str.startswith('Num')].to_list()

    pca = PCA(n_components=0.90, random_state=42)

    X_pca = pca.fit(df_new[pca_cols])

    loadings = X_pca.components_.T 

    loadings_df = pd.DataFrame(loadings, 
                           columns = [f'PC{i+1}' for i in range(X_pca.explained_variance_ratio_.shape[0])],
                           index=X_pca.feature_names_in_)
    
    for i in range(1):

        pc = []

        for col in loadings_df.index:

            df_new[f'PC{i+1}_{col}'] = df_new[col] * loadings_df.loc[col][f'PC{i+1}']

            pc.append(f'PC{i+1}_{col}')

        df_new[f'PC{i+1}'] = df_new[pc].sum(axis = 1)

        df_new = df_new.drop(columns = pc, axis = 1)

        del pc

    # Explained variation amongst variables
    print(f'The variance explained by PC1 {X_pca.explained_variance_ratio_[0] : .2%}')
    print(f'The variance explained by PC2 {X_pca.explained_variance_ratio_[1] : .2%}')
    print(f'The variance explained by PC3 {X_pca.explained_variance_ratio_[2] : .2%}')
    print('...')
    print(f'The variance explained by PC{X_pca.explained_variance_ratio_.shape[0]} {X_pca.explained_variance_ratio_[X_pca.explained_variance_ratio_.shape[0]-1] : .2%}')

    # Noise level of components
    print(f'The variance explained by last 2 components is {X_pca.explained_variance_ratio_[-2:].sum() : .2%}')


    print('PC feature engineering complete')
    print('///////////////////////////////')


    return df_new




In [ ]:
#Xtrain_final = pca_fe(Xtrain_final)

#Xtest_pre = pca_fe(Xtest_pre)

#Xtrain_final.shape, Xtest_pre.shape

In [ ]:
Xtrain_final.info()

# XgBoost Classifier #

In [ ]:

num_pre = Xtrain_pre.select_dtypes(['int','float']).columns.to_list()

cat_pre  = Xtrain_pre.select_dtypes('category').columns.to_list()
te_features = cat_pre

In [ ]:
te_features

In [ ]:
# Device agnostics

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
race_cols = list(Xtrain_final.Race.unique())

driver_cols = list(Xtrain_num.Driver.unique())



In [ ]:
Xtrain_final.Driver = Xtrain_final.Driver.astype('category')

Xtest_pre.Driver = Xtest_pre.Driver.astype('category')

In [ ]:
# Full model 

# OOF predictions
oof_preds = np.zeros(shape=(len(ytrain_final),2))

bagging_oof = np.zeros_like(test_id).astype('float64')

# Scoring
auc_score = pd.DataFrame(columns=['auc_score'])

# Group Stratified K-Fold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42

)


# Train/Test
for fold,(train_idx, val_idx) in enumerate(skf.split(X = Xtrain_final,y = ytrain_final)):

    print(f'Looping through fold:{fold}')

    X_train, X_val = Xtrain_final.iloc[train_idx], Xtrain_final.iloc[val_idx]

    y_train, y_val = ytrain_final.iloc[train_idx], ytrain_final.iloc[val_idx]

    # Taregt encoding 

    te = TargetEncoder(cv = 5,
                       target_type='binary',
                       random_state=42,
                       shuffle=True,
                       smooth='auto')
    
    X_train_te = te.fit_transform(X_train[te_features], y_train)

    X_val_te = te.transform(X_val[te_features])

    X_test = Xtest_pre.copy()

    X_test_te = te.transform(X_test[te_features])

    te_cols = []

    for var in te_features:

        te_cols.append(f'TE_{var}_encoded')

    X_train[te_cols] = X_train_te

    X_val[te_cols] = X_val_te

    X_test[te_cols] = X_test_te


        

    # Class weights
    cw = compute_class_weight(class_weight='balanced',
                              y = y_train,
                              classes = np.unique(y_train))
    
    print(f'Class Weights: {cw}')

    cw_dict = {
        0: cw.item(0),
        1: cw.item(1)
    }


    # Model
    model = lgb.LGBMClassifier(n_estimators = 200,
                              learning_rate = 0.1,
                              n_jobs = 1,
                              class_weight = cw_dict,
                              objective = 'binary',
                              random_state = 42,
                              metric = 'auc')

    # Train
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              categorical_feature=['Race','Year','Compound'])

     # Predictions 

    y_preds = model.predict_proba(X_val)[:, 1]

    oof_preds[val_idx] = model.predict_proba(X_val)

    bagging_oof += model.predict_proba(X_test)[:,1]/5

    # AUC
    score = roc_auc_score(y_val, y_preds)

    auc_score.loc[fold, 'auc_score'] = score

    del X_test, X_train_te, X_val_te, model, X_test_te




oof_score = roc_auc_score(ytrain_final, oof_preds[:, 1])
print(f'Out of fold predicition auc score: {oof_score: .5f}')


In [ ]:
# Partitioned year models 
year = list(Xtrain.Year.unique())

skf = StratifiedKFold(n_splits= 5,
                      random_state=42,
                      shuffle=True)


# OOF predictions
oof_preds_yr = np.zeros(shape=(len(ytrain_final),2))

bagging_oof_yr = np.zeros_like(test_id).astype('float64')



for i, col in enumerate(year):


    Xtr_yr = Xtrain_final[Xtrain_final.Year == col]

    Xts_yr = Xtest_pre[Xtest_pre.Year == col]

    ytr_yr = ytrain_final.loc[Xtr_yr.index]


    # save original indices
    yr_indices = ytr_yr.index
    test_indices = Xts_yr.index

    Xtr_yr = Xtr_yr.drop(columns=['Year'], axis = 1)
    Xts_yr = Xts_yr.drop(columns = ['Year'], axis = 1)

    te_features_yr = list(set(te_features) - set(['Year']))


    for fold, (train_idx, val_idx) in enumerate(skf.split(Xtr_yr, ytr_yr)):

        print(f'Looping through fold:{fold} and year:{col}')

        X_train = Xtr_yr.iloc[train_idx]
        X_val   = Xtr_yr.iloc[val_idx]

        y_train = ytr_yr.iloc[train_idx]
        y_val   = ytr_yr.iloc[val_idx]


        # Target encoding 

        te = TargetEncoder(cv = 5,
                           smooth = 'auto',
                           target_type = 'binary',
                           random_state=42,
                           shuffle = True)
        
        X_train_te = te.fit_transform(X_train[te_features_yr], y_train)

        X_val_te = te.transform(X_val[te_features_yr])

        X_test = Xts_yr.copy()

        X_test_te = te.transform(X_test[te_features_yr])

        te_cols_yr = []


        for var in te_features_yr:

            te_cols_yr.append(f'TE_{var}_encoded')

        X_train[te_cols_yr] = X_train_te

        X_val[te_cols_yr] = X_val_te

        X_test[te_cols_yr] = X_test_te


        # Class weights
        cw = compute_class_weight(class_weight='balanced',
                              y = y_train,
                              classes = np.unique(y_train))
    
        print(f'Class Weights: {cw}')

        cw_dict = {
        0: cw.item(0),
        1: cw.item(1)}

        model_yr = lgb.LGBMClassifier(n_estimators = 300,
                              learning_rate = 0.1,
                              n_jobs = -1,
                              class_weight = cw_dict,
                              objective = 'binary',
                              random_state = 42,
                              metric = 'auc')

        model_yr.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            categorical_feature=['Race','Compound']
        )

        y_preds_yr = model_yr.predict_proba(X_val)

        oof_preds_yr[yr_indices[val_idx]] = y_preds_yr


        bagging_oof_yr[test_indices] += model_yr.predict_proba(X_test)[:, 1]/5

        score = roc_auc_score(y_val, y_preds_yr[:, 1])

        print(f'Score : {score : .2f}')

        del X_test, X_test_te, X_train_te, X_val_te, model_yr, cw
    


raw_oof_score = roc_auc_score(ytrain_final, oof_preds_yr[:, 1])
print(f'Raw OOF score: {raw_oof_score: .5f}')

In [ ]:
# Partitioned compound models
compounds = list(Xtrain.Compound.unique())


splits_map = {
    'MEDIUM':5,
    'HARD':5,
    'SOFT':3,
    'INTERMEDIATE':3,
    'WET':2
}


# OOF predictions
oof_preds_comp = np.zeros(shape=(len(ytrain_final),2))

bagging_oof_comp = np.zeros_like(test_id).astype('float64')



for i, col in enumerate(compounds):


    Xtr_comp = Xtrain_final[Xtrain_final.Compound == col]

    Xts_comp = Xtest_pre[Xtest_pre.Compound == col]

    ytr_comp = ytrain_final.iloc[Xtr_comp.index]

    skf = StratifiedKFold(n_splits= splits_map[col],
                      random_state=42,
                      shuffle=True)

    # save original indices
    comp_indices = Xtr_comp.index
    test_indices = Xts_comp.index

    Xtr_comp = Xtr_comp.drop(columns=['Compound'])
    Xts_comp = Xts_comp.drop(columns = ['Compound'])

    te_features_comp = list(set(te_features) - set(['Compound']))

    

    for fold, (train_idx, val_idx) in enumerate(skf.split(Xtr_comp, ytr_comp)):

        print(f'Looping through fold:{fold} and compound:{col}')

        X_train = Xtr_comp.iloc[train_idx]
        X_val   = Xtr_comp.iloc[val_idx]

        y_train = ytr_comp.iloc[train_idx]
        y_val   = ytr_comp.iloc[val_idx]

        # Target encoding 

        te = TargetEncoder(cv = splits_map[col],
                           smooth = 'auto',
                           target_type = 'binary',
                           random_state=42,
                           shuffle = True)
        
        X_train_te = te.fit_transform(X_train[te_features_comp], y_train)

        X_val_te = te.transform(X_val[te_features_comp])

        X_test = Xts_comp.copy()

        X_test_te = te.transform(X_test[te_features_comp])

        te_cols_comp = []


        for var in te_features_comp:

            te_cols_comp.append(f'TE_{var}_encoded')

        X_train[te_cols_comp] = X_train_te

        X_val[te_cols_comp] = X_val_te

        X_test[te_cols_comp] = X_test_te


        # Class weights
        cw = compute_class_weight(class_weight='balanced',
                              y = y_train,
                              classes = np.unique(y_train))
    
        print(f'Class Weights: {cw}')

        cw_dict = {
        0: cw.item(0),
        1: cw.item(1)}

        model_comp = lgb.LGBMClassifier(n_estimators = 300,
                              learning_rate = 0.1,
                              n_jobs = -1,
                              class_weight = cw_dict,
                              objective = 'binary',
                              random_state = 42,
                              metric = 'auc')

        model_comp.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            categorical_feature= ['Race','Year']
        )

        y_preds_comp = model_comp.predict_proba(X_val)

        oof_preds_comp[comp_indices[val_idx]] = y_preds_comp

        bagging_oof_comp[test_indices] += model_comp.predict_proba(X_test)[:, 1]/splits_map[col]

        score = roc_auc_score(y_val, y_preds_comp[:, 1])

        print(f'Score : {score : .2f}')

        del X_test, X_test_te, X_train_te, X_val_te, model_comp, cw


raw_oof_score = roc_auc_score(ytrain_final, oof_preds_comp[:, 1])
print(f'Raw OOF score: {raw_oof_score: .5f}')

In [ ]:
# Partitioned race models 

skf = StratifiedKFold(n_splits= 5,
                      random_state=42,
                      shuffle=True)

# OOF predictions
oof_preds_race = np.zeros(shape=(len(ytrain_final),2))

bagging_oof_race = np.zeros_like(test_id).astype('float64')


for i, col in enumerate(race_cols):


    Xtr_race = Xtrain_final[Xtrain_final.Race == col]

    Xts_race = Xtest_pre[Xtest_pre.Race == col]

    ytr_race = ytrain_final.iloc[Xtr_race.index]

    # save original indices
    race_indices = Xtr_race.index
    test_indices = Xts_race.index

    Xtr_race = Xtr_race.drop(columns=['Race'])
    Xts_race = Xts_race.drop(columns = ['Race'])

    te_features_race = list(set(te_features) - set(['Race']))

    for fold, (train_idx, val_idx) in enumerate(skf.split(Xtr_race, ytr_race)):

        print(f'Looping through fold:{fold} and race:{col}')

        X_train = Xtr_race.iloc[train_idx]
        X_val   = Xtr_race.iloc[val_idx]

        y_train = ytr_race.iloc[train_idx]
        y_val   = ytr_race.iloc[val_idx]

        # Target encoding 

        te = TargetEncoder(cv = 5,
                           smooth = 'auto',
                           target_type = 'binary',
                           random_state=42,
                           shuffle = True)
        
        X_train_te = te.fit_transform(X_train[te_features_race], y_train)

        X_val_te = te.transform(X_val[te_features_race])

        X_test = Xts_race.copy()

        X_test_te = te.transform(X_test[te_features_race])

        te_cols_race = []


        for var in te_features_race:

            te_cols_race.append(f'TE_{var}_encoded')

        X_train[te_cols_race] = X_train_te

        X_val[te_cols_race] = X_val_te

        X_test[te_cols_race] = X_test_te


        # Class weights
        cw = compute_class_weight(class_weight='balanced',
                              y = y_train,
                              classes = np.unique(y_train))
    
        print(f'Class Weights: {cw}')

        cw_dict = {
        0: cw.item(0),
        1: cw.item(1)}

        model_race = lgb.LGBMClassifier(n_estimators = 300,
                              learning_rate = 0.1,
                              n_jobs = -1,
                              class_weight = cw_dict,
                              objective = 'binary',
                              random_state = 42,
                              metric = 'auc')

        model_race.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            categorical_feature=['Compound','Year']
        )

        y_preds_race = model_race.predict_proba(X_val)

        oof_preds_race[race_indices[val_idx]] = y_preds_race

        bagging_oof_race[test_indices] += model_race.predict_proba(X_test)[:, 1]/5

        score = roc_auc_score(y_val, y_preds_race[:, 1])

        print(f'Score : {score : .2f}')

        del X_test, X_test_te, X_train_te, X_val_te, model_race, cw

raw_oof_score = roc_auc_score(ytrain_final, oof_preds_race[:, 1])
print(f'Raw OOF score: {raw_oof_score: .5f}')

# Final prediction models # 

In [ ]:
# Full model

cw = compute_class_weight(y = ytrain_final,
                          class_weight='balanced',
                          classes = np.unique(ytrain_final))

cw_dict = {
    0: cw.item(0),
    1: cw.item(1)
}

model_full = lgb.LGBMClassifier(n_estimators = 300,
                              learning_rate = 0.1,
                              n_jobs = -1,
                              class_weight = cw_dict,
                              objective = 'binary',
                              random_state = 42,
                              metric = 'auc')

model_full.fit(Xtrain_final, ytrain_final, categorical_feature=['Race','Year','Compound'])

full_preds = model_full.predict_proba(Xtest_pre)[:, 1]

full_preds = pd.DataFrame(full_preds, columns = ['y1'])

In [ ]:
# Year models 

year_preds = pd.DataFrame(np.zeros_like(test_id))


for yr in year:

    # Partitioning data by year

    Xtr_yr = Xtrain_final[Xtrain_final.Year == yr]

    Xtr_yr_idx = Xtr_yr.index

    Xts_yr = Xtest_pre[Xtest_pre.Year == yr]

    Xts_yr_idx = Xts_yr.index

    ytr_yr = ytrain_final[Xtr_yr_idx]

    # Class weights 

    cw = compute_class_weight(y = ytr_yr,
                          class_weight='balanced',
                          classes = np.unique(ytrain_final))

    cw_dict_yr = {
    0: cw.item(0),
    1: cw.item(1)
    }

    # Model 

    model_yr = lgb.LGBMClassifier(n_estimators = 300,
                              learning_rate = 0.1,
                              n_jobs = -1,
                              class_weight = cw_dict_yr,
                              objective = 'binary',
                              random_state = 42,
                              metric = 'auc')
    

    model_yr.fit(Xtr_yr, ytr_yr, categorical_feature=['Race','Year','Compound'])

    year_preds.iloc[Xts_yr_idx] = pd.DataFrame(model_yr.predict_proba(Xts_yr)[:, 1])

    print(f'Completed predcitions for year : {yr}')

In [ ]:
# Compund models

compound_preds = pd.DataFrame(np.zeros_like(test_id))

for comp in compounds:

    # Partitioning data by compound

    Xtr_comp = Xtrain_final[Xtrain_final.Compound == comp]

    Xtr_comp_idx = Xtr_comp.index

    Xts_comp = Xtest_pre[Xtest_pre.Compound == comp]

    Xts_comp_idx = Xts_comp.index

    ytr_comp = ytrain_final[Xtr_comp_idx]

    # Class weights 

    cw = compute_class_weight(y = ytr_comp,
                          class_weight='balanced',
                          classes = np.unique(ytrain_final))

    cw_dict_comp = {
    0: cw.item(0),
    1: cw.item(1)
    }

    # Model 

    model_comp = lgb.LGBMClassifier(n_estimators = 300,
                              learning_rate = 0.1,
                              n_jobs = -1,
                              class_weight = cw_dict_comp,
                              objective = 'binary',
                              random_state = 42,
                              metric = 'auc')
    

    model_comp.fit(Xtr_comp, ytr_comp, categorical_feature=['Race','Compound','Year'])

    compound_preds.iloc[Xts_comp_idx] = pd.DataFrame(model_comp.predict_proba(Xts_comp)[:, 1])

    print(f'Completed predcitions for compound : {comp}')

In [ ]:
# Race models

race_preds = pd.DataFrame(np.zeros_like(test_id))

for race in race_cols:

    # Partitioning data by race

    Xtr_race = Xtrain_final[Xtrain_final.Race == race]

    Xtr_race_idx = Xtr_race.index

    Xts_race = Xtest_pre[Xtest_pre.Race == race]

    Xts_race_idx = Xts_race.index

    ytr_race = ytrain_final[Xtr_race_idx]

    # Class weights

    cw = compute_class_weight(y = ytr_race,
                          class_weight='balanced',
                          classes = np.unique(ytrain_final))

    cw_dict_race = {
    0: cw.item(0),
    1: cw.item(1)
    }


    model_race = lgb.LGBMClassifier(n_estimators = 300,
                              learning_rate = 0.1,
                              n_jobs = -1,
                              class_weight = cw_dict_race,
                              objective = 'binary',
                              random_state = 42,
                              metric = 'auc')
    

    model_race.fit(Xtr_race, ytr_race, categorical_feature=['Race','Year','Compound'])

    race_preds.iloc[Xts_race_idx] = pd.DataFrame(model_race.predict_proba(Xts_race)[:, 1])

    print(f'Completed predcitions for race : {race}')



In [ ]:
full_preds.shape, race_preds.shape, compound_preds.shape, year_preds.shape



In [ ]:
training_preds = np.c_[oof_preds[:, 1], oof_preds_yr[:, 1], oof_preds_comp[:, 1], oof_preds_race[:, 1]]

training_preds = pd.DataFrame(training_preds, columns = ['y1','y2','y3','y4'])

training_preds

In [ ]:
test_preds = pd.concat([full_preds, year_preds, compound_preds, race_preds], axis = 1)

test_preds.columns = ['y1','y2','y3','y4']

test_preds

In [ ]:
means_test = test_preds.mean(axis = 1)
means_test

In [ ]:
test_oof = np.c_[bagging_oof, bagging_oof_yr, bagging_oof_comp, bagging_oof_race]
test_oof = pd.DataFrame(test_oof, columns = ['y1','y2','y3','y4'])
test_oof

In [ ]:
# Meta model 

calib = CalibratedClassifierCV(method='isotonic',
                               cv = 5,
                               ensemble='auto')

calib.fit(training_preds, ytrain_final)

prob_preds = calib.predict_proba(test_preds)[:, 1]




In [ ]:
prob_preds

In [ ]:
means_oof = test_oof.mean(axis = 1)

means_oof

In [ ]:
test_oof = pd.DataFrame(means_oof, columns = ['PitNextLap'])
test_final = pd.DataFrame(means_test, columns = ['PitNextLap'])
prob_preds_final = pd.DataFrame(prob_preds, columns = ['PitNextLap'])
test_oof.shape, test_final.shape, prob_preds_final.shape

In [ ]:
mask = (test_oof['PitNextLap'] <= 0.7) & (test_oof['PitNextLap'] >= 0.4)
mask

In [ ]:
unconfident_preds = Xtest_freq[mask]

In [ ]:
unconfident_preds.describe(include='object')

In [ ]:
submission = pd.concat([test_id,prob_preds_final['PitNextLap']], axis = 1)

In [ ]:
submission.to_csv('Catmods_08.csv', index=False)